# HW3 — Car Class with Iterators, OOP & Weather API

In [1]:
!pip install openmeteo_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.6/718.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.9 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [2]:
import openmeteo_requests
from datetime import datetime

## Iterator Classes

In [3]:
class IncreaseSpeed:
    '''
    Iterator for increasing the speed with the default step of 10 km/h.
    Composition approach: initialized inside Car, bound to a specific Car instance.

    Constructor params:
      current_speed: a value to start with, km/h
      max_speed: a maximum possible value, km/h
      step: increment step (default 10)
    '''

    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration
        self.current_speed = min(self.current_speed + self.step, self.max_speed)
        return self.current_speed


class DecreaseSpeed:
    '''
    Iterator for decreasing the speed with the default step of 10 km/h.
    Composition approach: initialized inside Car, bound to a specific Car instance.

    Constructor params:
      current_speed: a value to start with, km/h
      min_speed: a minimum possible value (floor), km/h
      step: decrement step (default 10)
    '''

    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        self.min_speed = max(min_speed, 0)  # never below zero
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration
        self.current_speed = max(self.current_speed - self.step, self.min_speed)
        return self.current_speed

## Car Class

In [4]:
class Car:
    '''
    Car class.
    Has a class variable for counting total cars on the road.
    A car with current_speed == 0 starts off the road.

    Constructor params:
      max_speed: maximum possible speed, km/h
      current_speed: current speed, km/h (0 by default → off road)
    '''

    _cars_on_road = 0  # class variable: counts cars currently on the road

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed

        # A car is on the road only if it has non-zero speed at creation
        if current_speed > 0:
            self.state = 'on road'
            Car._cars_on_road += 1
        else:
            self.state = 'off road'

    # Regular method — works with instance state

    def accelerate(self, upper_border=None, step=10):
        '''
        Increases the speed using IncreaseSpeed iterator.
        If the car is off road, puts it on the road first.
        If upper_border is given and valid, increases gradually to that value.
        Otherwise increases once by one step.
        '''
        # If the car is off road, bring it onto the road
        off_road = self.state == 'off road'
        if off_road:
            self.state = 'on road'
            Car._cars_on_road += 1

        speed_before = self.current_speed

        if upper_border is not None and 0 < upper_border <= self.max_speed:
            # Gradually accelerate to upper_border
            increaser = IncreaseSpeed(self.current_speed, upper_border, step)
            for speed in increaser:
                if not off_road:  # skip step-by-step messages when coming off parking
                    print('INFO: Speed increases by', step)
                self.current_speed = speed
        else:
            # Increase once
            increaser = IncreaseSpeed(self.current_speed, self.max_speed, step)
            self.current_speed = next(increaser)

        print(f'INFO: The speed of this car has been increased from {speed_before} to {self.current_speed}')

    def brake(self, lower_border=None, step=10):
        '''
        Decreases the speed using DecreaseSpeed iterator.
        If lower_border is given and valid, decreases gradually to that value.
        Otherwise decreases once by one step.
        '''
        speed_before = self.current_speed

        if lower_border is not None and 0 <= lower_border < self.current_speed:
            # Gradually brake to lower_border
            decreaser = DecreaseSpeed(self.current_speed, lower_border, step)
            for speed in decreaser:
                print('INFO: Speed decreases by', step)
                self.current_speed = speed
        else:
            # Decrease once
            decreaser = DecreaseSpeed(self.current_speed, 0, step)
            self.current_speed = next(decreaser)

        print(f'INFO: The speed of this car has been decreased from {speed_before} to {self.current_speed}')

    def parking(self):
        '''
        Regular method.
        Moves the car to parking (off road).
        Ensures speed is 0 and updates the class counter.
        Cannot park a car that is already off road.
        '''
        if self.state == 'off road':
            print('INFO: The car is already in the parking.')
            return

        # Bring speed to 0 if not already
        speed_before = self.current_speed
        self.current_speed = 0
        print(f'INFO: The speed of this car has been decreased from {speed_before} to {self.current_speed}')

        self.state = 'off road'
        Car._cars_on_road -= 1
        print('Parking the car...')

    # Class method — works with class state


    @classmethod
    def total_cars(cls):
        '''
        Classmethod.
        Returns the total number of cars currently on the road.
        Called as Car.total_cars().
        '''
        return cls._cars_on_road

    # Static method — no access to instance or class state needed

    @staticmethod
    def show_weather():
        '''
        Staticmethod.
        Fetches and displays current weather conditions via Open-Meteo API.
        Can be called both on instance (car.show_weather()) and on class (Car.show_weather()).
        '''
        openmeteo = openmeteo_requests.Client()
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {
            'latitude': 59.9386,   # St. Petersburg
            'longitude': 30.3141,
            'current': [
                'temperature_2m',
                'apparent_temperature',
                'rain',
                'wind_speed_10m'
            ],
            'wind_speed_unit': 'ms',
            'timezone': 'Europe/Moscow'
        }

        response = openmeteo.weather_api(url, params=params)[0]
        current = response.Current()

        temperature    = current.Variables(0).Value()
        apparent_temp  = current.Variables(1).Value()
        rain           = current.Variables(2).Value()
        wind_speed     = current.Variables(3).Value()

        print(f'Current temperature: {round(temperature, 1)} C')
        print(f'Current apparent_temperature: {round(apparent_temp, 1)} C')
        print(f'Current rain: {rain} mm')
        print(f'Current wind_speed: {round(wind_speed, 1)} m/s')

## Tests

In [5]:
car1 = Car(100, 20)   # max_speed=100, starts on road at 20
car2 = Car(60, 30)    # max_speed=60,  starts on road at 30
car3 = Car(100, 0)    # speed=0 → starts off road

print(f'Total cars on road: {Car.total_cars()}')  # expects 2

Total cars on road: 2


In [6]:
car1.accelerate(100)  # 20 → 100, step-by-step

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [7]:
car2.accelerate(50)   # 30 → 50, step-by-step

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [8]:
print('Speed of car 1:', car1.current_speed)  # expects 100
print('Speed of car 2:', car2.current_speed)  # expects 50

Speed of car 1: 100
Speed of car 2: 50


In [9]:
car1.brake(10)   # 100 → 10, step-by-step

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [10]:
car2.brake(0)                              # 50 → 0
print('Total cars on road:', Car.total_cars())  # expects 2
car2.parking()                             # goes off road
print('Total cars on road:', Car.total_cars())  # expects 1

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [11]:
car3.accelerate(80)          # off road → on road, 0 → 80
car3.show_weather()          # instance call
print('Total cars on road:', Car.total_cars())  # expects 2

INFO: The speed of this car has been increased from 0 to 80
Current temperature: 3.8 C
Current apparent_temperature: 0.6 C
Current rain: 0.0 mm
Current wind_speed: 2.3 m/s
Total cars on road: 2


In [12]:
car2.accelerate(10)          # parking → on road, 0 → 10
print('Total cars on road:', Car.total_cars())  # expects 3

INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [13]:
Car.show_weather()           # class call

Current temperature: 3.8 C
Current apparent_temperature: 0.6 C
Current rain: 0.0 mm
Current wind_speed: 2.3 m/s
